In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [2]:
#@title 🔄 Colab & Local Jupyter Folder Switcher (click to expand)

# 🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
# 🚨        Change only this 🚨
relative_path = "_datasets" #🚨 any relative path
# 🚨        Change only this 🚨
# 🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨

import os

create_if_missing = True      # False = error if folder missing

def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def paths_equal(a: str, b: str) -> bool:
    """Portable path equality (no os.path.samepath)."""
    return os.path.normcase(os.path.realpath(a)) == os.path.normcase(os.path.realpath(b))

def resolve_local_base(rel_path: str) -> str:
    pinned = os.getenv("NOTEBOOK_ROOT")
    if pinned:
        return pinned
    cwd = os.path.abspath(os.getcwd())
    rel_name = os.path.basename(os.path.normpath(rel_path))
    base = os.path.dirname(cwd) if rel_name and os.path.basename(cwd) == rel_name else cwd
    os.environ["NOTEBOOK_ROOT"] = base
    return base

def resolve_colab_base(rel_path: str) -> str:
    pinned = os.getenv("COLAB_ROOT")
    if pinned:
        return pinned
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    mydrive = "/content/drive/MyDrive"
    colab_nb = os.path.join(mydrive, "Colab Notebooks")
    base = colab_nb if os.path.isdir(colab_nb) else mydrive
    os.environ["COLAB_ROOT"] = base
    return base

# --- Pick base path ---
if in_colab():
    print("✅ Detected Google Colab")
    base_path = resolve_colab_base(relative_path)
else:
    print("✅ Detected Local Jupyter")
    base_path = resolve_local_base(relative_path)

# --- Build target path ---
if os.path.isabs(relative_path):
    target_path = os.path.abspath(relative_path)
else:
    target_path = os.path.abspath(os.path.join(base_path, relative_path))

current = os.path.abspath(os.getcwd())
rel_name = os.path.basename(os.path.normpath(relative_path))
if rel_name and os.path.basename(current) == rel_name:
    target_path = current

# --- Ensure exists ---
if not os.path.exists(target_path):
    if create_if_missing:
        os.makedirs(target_path, exist_ok=True)
        print(f"📁 Created folder: {target_path}")
    else:
        raise FileNotFoundError(f"❌ Path does not exist: {target_path}")

# --- Change only if needed ---
if paths_equal(current, target_path):
    print(f"📂 Already working in: {current}")
else:
    os.chdir(target_path)
    print(f"📂 Now working in: {os.path.abspath(os.getcwd())}")


✅ Detected Local Jupyter
📂 Now working in: /Users/jayklarin/Documents/__DI/Repositories/di_bootcamp/Colab-Swap/_datasets


# Exercises XP Gold

## 👩‍🏫 👩🏿‍🏫 What You’ll Learn

* Perform a comparative analysis of structured and unstructured retail data.
* Understand the processing and analysis of structured and unstructured healthcare data.
* Gain familiarity with structured data through basic exploration.
* Understand the challenges of working with unstructured data and identify structured elements in a public transportation dataset.
* Generate a synthetic product catalog for an e-commerce platform using Faker.


## 🛠️ What You Will Create

* A comparative analysis document discussing the insights from structured and unstructured retail data and the challenges in processing them.
* Identification of structured data elements within the E-Commerce dataset.
* Categorization of data elements in the dataset as structured or unstructured with justifications.
* A Python script using Faker to generate a synthetic product catalog for an e-commerce platform.

## Exercise 1: Comparative Analysis Of Retail Data

Dataset: Use the Retail Dataset for structured data and Women’s E-Commerce Clothing Reviews for unstructured data.

* Analyze the Retail Dataset focusing on sales trends, customer purchase patterns, and store performance.
* Analyze the Clothing Reviews dataset to extract insights like predominant sentiments, frequently mentioned topics, and overall customer satisfaction.
* Compare the insights you can derive from each dataset and discuss the challenges you faced in processing the unstructured data.

In [5]:
features_df = pd.read_csv('_Features data set.csv')
#features_df.info()

In [6]:
sales_df = pd.read_csv('_sales_data_set.csv')
#sales_df.info()

In [7]:
stores_df = pd.read_csv('_stores_data_set.csv')
#stores_df.info()

In [8]:
retail_df = pd.read_csv('womens_clothing_e_commerce_reviews.csv')
#retail_df.info()

## 📊 Part A — Structured Retail Data

In [10]:
# --------- 1) ROBUST DATE PARSING & MERGE ---------
# Some rows are dd/mm/yyyy and others mm/dd/yyyy; handle both without warnings.

for _df in [features_df, sales_df]:
    s = _df["Date"].astype(str)

    # Pass 1: day-first
    d = pd.to_datetime(s, format="mixed", dayfirst=True, errors="coerce")

    # Pass 2: month-first for any unparsed rows
    bad = d.isna()
    if bad.any():
        d.loc[bad] = pd.to_datetime(s[bad], format="mixed", dayfirst=False, errors="coerce")

    # Normalize to midnight (removes any time components)
    _df["Date"] = d.dt.normalize()

# Build the merged "retail" table (sales + features + stores)
retail = (sales_df
          .merge(features_df, on=["Store", "Date"], how="left", suffixes=("", "_feat"))
          .merge(stores_df, on="Store", how="left"))

# Calendar helpers (no warnings)
retail = retail.assign(
    Year = retail["Date"].dt.year,
    Week = retail["Date"].dt.isocalendar().week.astype(int),  # ensure plain int if needed later
    Month = retail["Date"].dt.month
)

# Quick sanity check
assert {"Date","Weekly_Sales","Store"}.issubset(retail.columns), "Merged 'retail' missing required columns."

In [11]:
# --------- 2) SALES TRENDS ---------
# Company-wide weekly sales with a smooth rolling average — using .ffill() (no FutureWarning).

weekly_total = (retail
                .groupby("Date", as_index=True)["Weekly_Sales"]
                .sum())

# Ensure a proper weekly DateTimeIndex and forward-fill gaps
weekly_total = (weekly_total
                .asfreq("W")   # converts to weekly index
                .ffill())      # forward fill without deprecated 'method=' syntax

# 8-week moving average
trend = weekly_total.rolling(8, min_periods=1).mean()

print("Last 3 weeks (raw):")
print(weekly_total.tail(3))
print("\nLast 3 weeks (8-wk avg):")
print(trend.tail(3))

# Holiday vs Non-Holiday average sales
holiday_effect = retail.groupby("IsHoliday", observed=True)["Weekly_Sales"].mean()
print("\nHoliday Sales Lift (avg weekly sales):")
print(holiday_effect)

Last 3 weeks (raw):
Date
2012-10-07   NaN
2012-10-14   NaN
2012-10-21   NaN
Freq: W-SUN, Name: Weekly_Sales, dtype: float64

Last 3 weeks (8-wk avg):
Date
2012-10-07   NaN
2012-10-14   NaN
2012-10-21   NaN
Freq: W-SUN, Name: Weekly_Sales, dtype: float64

Holiday Sales Lift (avg weekly sales):
IsHoliday
False    15901.445069
True     17035.823187
Name: Weekly_Sales, dtype: float64


In [12]:
# --------- 3) CUSTOMER PURCHASE PATTERNS ---------
# a) Top departments by total sales
dept_totals = (retail.groupby("Dept", observed=True)["Weekly_Sales"]
               .sum()
               .sort_values(ascending=False)
               .head(10))
print("\nTop Departments by Total Sales:")
print(dept_totals)

# b) Department seasonality by month
dept_month = (retail.groupby(["Dept","Month"], observed=True)["Weekly_Sales"]
              .mean()
              .unstack("Month", fill_value=0))
print("\nDepartment Seasonality (mean weekly sales by month):")
print(dept_month.head())

# c) Simple promotion signal: correlation between markdowns and sales
mark_cols = ["MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5"]
# numeric_only=True avoids dtype warnings when non-numeric cols sneak in
corr_md = retail[mark_cols + ["Weekly_Sales"]].corr(numeric_only=True)["Weekly_Sales"].drop("Weekly_Sales")
print("\nCorrelation of MarkDowns with Weekly_Sales:")
print(corr_md.sort_values(ascending=False))


Top Departments by Total Sales:
Dept
92    4.839433e+08
95    4.493202e+08
38    3.931181e+08
72    3.057252e+08
90    2.910685e+08
40    2.889360e+08
2     2.806112e+08
91    2.167817e+08
13    1.973216e+08
8     1.942808e+08
Name: Weekly_Sales, dtype: float64

Department Seasonality (mean weekly sales by month):
Month            1             2             3             4             5   \
Dept                                                                          
1      13665.363194  23439.498833  18478.219556  27969.288698  14168.030111   
2      39847.003639  43320.360593  43031.601094  42808.345698  42881.651852   
3      11149.947111   9464.377815   8296.513162   7961.761444   7583.878241   
4      25369.277750  25325.666222  24748.209487  25080.069127  25509.505463   
5      16895.285534  24145.523032  22450.608656  20723.475761  15088.308327   

Month            6             7             8             9             10  \
Dept                                              

In [13]:
# --------- 4) STORE PERFORMANCE ---------
store_perf = (retail.groupby(["Store","Type","Size"], observed=True)["Weekly_Sales"]
              .sum()
              .reset_index()
              .sort_values("Weekly_Sales", ascending=False))

# Normalize by size (sales per sq ft)
store_perf["Sales_per_sqft"] = store_perf["Weekly_Sales"] / store_perf["Size"]

# Type-level summary (no deprecated agg usage)
type_summary = (store_perf.groupby("Type", observed=True)
                .agg(
                    total_sales = ("Weekly_Sales","sum"),
                    avg_sales_per_store = ("Weekly_Sales","mean"),
                    median_sales_per_sqft = ("Sales_per_sqft","median")
                )
               )

print("\nTop Stores (absolute):")
print(store_perf.head(5))
print("\nStore Type Summary:")
print(type_summary)


Top Stores (absolute):
    Store Type    Size  Weekly_Sales  Sales_per_sqft
19     20    A  203742  3.013978e+08     1479.311053
3       4    A  205863  2.995440e+08     1455.064550
13     14    A  200898  2.889999e+08     1438.540510
12     13    A  219622  2.865177e+08     1304.594730
1       2    A  202307  2.753824e+08     1361.210640

Store Type Summary:
       total_sales  avg_sales_per_store  median_sales_per_sqft
Type                                                          
A     4.331015e+09         1.968643e+08             996.757827
B     2.000701e+09         1.176883e+08            1268.512766
C     4.055035e+08         6.758392e+07            1659.095591


## 📝 Part B — Unstructured Reviews (Women’s E-Commerce Clothing)

In [15]:
# --------- 1) PREP TEXT DATA ---------
reviews = retail_df.copy()
reviews = reviews.rename(columns={
    "Review Text":"review_text",
    "Rating":"rating",
    "Department Name":"department"
})

# Keep only rows with text
reviews = reviews.loc[reviews["review_text"].notna()].copy()

# Optional: keep only plausible ratings (1..5) to avoid oddities in the dataset
reviews = reviews.loc[reviews["rating"].between(1,5)]

In [16]:
# --------- 2) SENTIMENT ANALYSIS (VADER) ---------
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download("vader_lexicon", quiet=True)

sia = SentimentIntensityAnalyzer()

# Vectorized apply with a lambda; no deprecated API here
reviews["sent_compound"] = reviews["review_text"].apply(lambda t: sia.polarity_scores(t)["compound"])

def bucket_score(c):
    if c >= 0.05: return "positive"
    if c <= -0.05: return "negative"
    return "neutral"

reviews["sentiment"] = reviews["sent_compound"].apply(bucket_score)

# Overall sentiment mix
sent_mix = reviews["sentiment"].value_counts(normalize=True).mul(100).round(1)
print("Sentiment Mix (%):")
print(sent_mix)

# Satisfaction snapshot (ratings + recommendations + feedback)
sat_snapshot = reviews.agg(
    avg_rating = ("rating","mean"),
    pct_recommended = ("Recommended IND", lambda s: float(s.mean()*100.0)),
    avg_pos_feedback = ("Positive Feedback Count","mean")
)
print("\nSatisfaction Snapshot:")
print(sat_snapshot)

# Department-level view
dept_sat = (reviews.groupby("department", dropna=True, observed=True)
            .agg(
                avg_rating=("rating","mean"),
                pct_recommended=("Recommended IND", lambda s: float(s.mean()*100.0)),
                avg_sent=("sent_compound","mean")
            )
            .sort_values(["avg_rating","avg_sent"], ascending=False))
print("\nDepartment Satisfaction (top 5):")
print(dept_sat.head(5))

Sentiment Mix (%):
sentiment
positive    92.7
negative     6.1
neutral      1.2
Name: proportion, dtype: float64

Satisfaction Snapshot:
                    rating  Recommended IND  Positive Feedback Count
avg_rating        4.183561              NaN                      NaN
pct_recommended        NaN        81.886842                      NaN
avg_pos_feedback       NaN              NaN                 2.630582

Department Satisfaction (top 5):
            avg_rating  pct_recommended  avg_sent
department                                       
Bottoms       4.278809        84.953577  0.747695
Intimate      4.271022        84.633999  0.725749
Jackets       4.254491        83.333333  0.737694
Tops          4.157743        81.070860  0.735760
Dresses       4.138812        80.520749  0.734405


In [17]:
# --------- 3) TOPIC MODELING (LDA) ---------
# Clean text without using deprecated sklearn APIs.
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

texts = reviews["review_text"].astype(str).map(clean_text)

# CountVectorizer: use get_feature_names_out() later (not deprecated)
vec = CountVectorizer(min_df=20, stop_words="english", max_features=5000, ngram_range=(1,2))
X = vec.fit_transform(texts)

# LDA (current API)
lda = LatentDirichletAllocation(n_components=5, random_state=0, learning_method="batch")
W = lda.fit_transform(X)        # document-topic weights
H = lda.components_             # topic-term weights
vocab = np.array(vec.get_feature_names_out())

def top_terms(H_row, k=10):
    # No deprecated args; uses numpy argsort as usual
    return ", ".join(vocab[np.argsort(H_row)[-k:][::-1]])

topics = [top_terms(H[i], 10) for i in range(H.shape[0])]
print("\nDiscovered Topics:")
for i, t in enumerate(topics, 1):
    print(f"Topic {i}: {t}")

# Dominant topic per review (optional downstream grouping)
reviews["topic_id"] = np.argmax(W, axis=1)


Discovered Topics:
Topic 1: dress, love, wear, flattering, perfect, beautiful, comfortable, great, compliments, fabric
Topic 2: size, small, true, fit, true size, large, ordered, little, runs, color
Topic 3: like, look, fit, love, just, fabric, pants, jeans, color, great
Topic 4: love, great, wear, soft, sweater, shirt, color, perfect, colors, comfortable
Topic 5: small, fit, petite, xs, like, ordered, size, just, lbs, medium


## 🔍 Part C — Compare & Reflect (Structured vs Unstructured)
What the structured Retail data tells you (the what):
* Trends: Weekly sales trajectory with clear holiday lift.
* Purchase patterns: Top-selling departments and seasonality by month.
* Store performance: Winners in absolute sales vs sales per sq ft, and differences by store type.
* Promotions: Basic signal that MarkDowns correlate with sales (strength varies by markdown).

What the unstructured Reviews add (the why):
* Sentiment: Share of positive/neutral/negative opinions; average rating and recommendation rate.
* Themes: Topics like fit/sizing, material/quality, style, price/value, shipping/returns.
* Department nuance: Which departments have stronger sentiment and satisfaction.

Challenges unique to unstructured text (and how we addressed them):
* Inconsistent writing & noise (typos, emojis, punctuation) → clean_text function.
* Ambiguity/sarcasm → VADER is good but imperfect; treat sentiment as directional, not absolute truth.
* Domain vocabulary (e.g., “runs small”, “sheer”) → n-grams and higher min_df to surface stable phrases.
* Imbalance (more positive than negative) → report percentages, not just counts.
* Model sensitivity (LDA depends on preprocessing) → fixed min_df, capped vocabulary, n_components chosen modestly (5).

How they complement each other:
* Use structured KPIs to locate where/when to focus (depts, stores, time windows).
* Use reviews to explain why outcomes occur and how to act (fit guidance, material improvements, sizing charts, messaging).

## Exercise 2: Basic Data Exploration In E-Commerce

Dataset: Use the “E-Commerce Data” dataset.

* Load the dataset using Pandas and print the first few rows to understand its structure.
* Print basic information about the dataset, like the number of rows, columns, and column names.
* Identify which columns in the dataset represent structured data (like numerical values, dates, fixed categories).
* Suggest what type of unstructured data could complement this dataset for a more comprehensive analysis (e.g., customer reviews, product descriptions).
* Discuss how this unstructured data might be used in conjunction with the structured data.

In [49]:
df = pd.read_csv('ecommerce_dataset.csv', encoding="latin1")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [53]:
# Number of rows
len(df) 

541909

In [57]:
# Number of columns
col_names = df.columns.tolist()
len(col_names)

8

In [59]:
# Column Names
col_names

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

## Exercise 3: Analyzing A Public Transportation Dataset With A Focus On Data Types

Dataset: Use the “Metro Interstate Traffic Volume” dataset.

* Load the dataset and display the first few rows to get a sense of the data.
* Identify and print the structured elements in the dataset, such as date-time, traffic volume, etc.
* Based on your observation of the dataset, categorize the data elements as structured or unstructured. For example, consider elements like weather descriptions, date-time, and traffic volume. Explain why you categorized them as such.

## Exercise 4: Basic Data Analysis In A Movie Ratings Dataset

Dataset: Use the “MovieLens Latest Datasets”.

* Load the ‘ratings.csv’ file from the dataset, which contains user ratings for movies, and display the first few rows.
* Identify and list down the structured elements in the dataset, such as user IDs, movie IDs, ratings, and timestamps.
* Explain why the data elements in the ‘ratings.csv’ file are considered structured data.

## Exercise 5: Creating A Synthetic Product Catalog

We want to generate a synthetic product catalog for an e-commerce platform using Faker.

* Ensure that the Faker library is installed and imported into your Python environment.
* Create a dataset of 500 products. Each product should have a unique ID, name, description, and price.
* Use Pandas to create a DataFrame from the generated data.